In [1]:
!pip install openai langchain langchain-openai requests

In [2]:


import os
import re
import json
import requests
from datetime import datetime
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

print("✓ Cliente OpenAI configurado correctamente.")

✓ Cliente OpenAI configurado correctamente.


In [3]:


import os
import re
import json
import requests
from datetime import datetime
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

print("✓ Cliente OpenAI configurado correctamente.")

✓ Cliente OpenAI configurado correctamente.


In [4]:


CENTROS = {
    "ensenada": {"lat": -41.140459, "lon": -72.404236, "nombre": "Piscicultura Petrohué"},
    "puelche":  {"lat": -41.733,    "lon": -73.602,    "nombre": "Centro Puelche"},
    "huito":    {"lat": -41.783,    "lon": -73.583,    "nombre": "Centro Huito (San José)"}
}

def get_clima_actual(args):
    """Obtiene el clima actual para un centro de cultivo de Camanchaca."""
    centro = args.get("centro", "ensenada").lower()
    
    if centro not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."
    
    datos = CENTROS[centro]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        current = data["current"]
        
        temp    = current["temperature_2m"]
        viento  = current["wind_speed_10m"]
        lluvia  = current["precipitation"]
        codigo  = current["weathercode"]
        
        condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
        
        return (
            f"Centro: {datos['nombre']}\n"
            f"Temperatura: {temp}°C\n"
            f"Viento: {viento} km/h\n"
            f"Precipitación: {lluvia} mm\n"
            f"Condición: {condicion}"
        )
    except Exception as e:
        return f"Error al obtener datos climáticos: {e}"


def get_pronostico_semana(args):
    """Obtiene el pronóstico de 7 días para un centro de cultivo de Camanchaca."""
    centro = args.get("centro", "ensenada").lower()
    
    if centro not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."
    
    datos = CENTROS[centro]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        daily = data["daily"]
        
        resultado = f"Pronóstico 7 días - {datos['nombre']}:\n"
        for i in range(7):
            fecha   = daily["time"][i]
            tmax    = daily["temperature_2m_max"][i]
            tmin    = daily["temperature_2m_min"][i]
            lluvia  = daily["precipitation_sum"][i]
            viento  = daily["wind_speed_10m_max"][i]
            codigo  = daily["weathercode"][i]
            
            condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
            resultado += (
                f"\n{fecha}: {tmin}°C - {tmax}°C | "
                f"Viento: {viento} km/h | "
                f"Lluvia: {lluvia} mm | {condicion}"
            )
        return resultado
    except Exception as e:
        return f"Error al obtener pronóstico: {e}"


def evaluar_condiciones_operacion(args):
    """Evalúa si las condiciones climáticas son seguras para operar en un centro."""
    centro = args.get("centro", "ensenada").lower()
    operacion = args.get("operacion", "cosecha")
    
    if centro not in CENTROS:
        return f"Centro '{centro}' no encontrado."
    
    datos = CENTROS[centro]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        current = data["current"]
        
        viento  = current["wind_speed_10m"]
        lluvia  = current["precipitation"]
        temp    = current["temperature_2m"]
        
        alertas = []
        if viento > 40:
            alertas.append(f" Viento peligroso: {viento} km/h (límite: 40 km/h)")
        if lluvia > 10:
            alertas.append(f" Lluvia intensa: {lluvia} mm")
        if temp < 5:
            alertas.append(f" Temperatura muy baja: {temp}°C")
        if temp > 18:
            alertas.append(f" Temperatura elevada: {temp}°C (riesgo para FCR)")
        
        if not alertas:
            estado = f" Condiciones APTAS para {operacion} en {datos['nombre']}."
        else:
            estado = f" Condiciones NO APTAS para {operacion} en {datos['nombre']}:\n" + "\n".join(alertas)
        
        return estado
    except Exception as e:
        return f"Error al evaluar condiciones: {e}"


tools = {
    "get_clima_actual": {
        "function": get_clima_actual,
        "description": "Obtiene el clima actual (temperatura, viento, lluvia) para un centro de cultivo de Camanchaca.",
        "args": {"centro": "nombre del centro: ensenada, puelche o huito"}
    },
    "get_pronostico_semana": {
        "function": get_pronostico_semana,
        "description": "Obtiene el pronóstico climático de 7 días para un centro de cultivo.",
        "args": {"centro": "nombre del centro: ensenada, puelche o huito"}
    },
    "evaluar_condiciones_operacion": {
        "function": evaluar_condiciones_operacion,
        "description": "Evalúa si las condiciones climáticas son seguras para realizar una operación (cosecha, biometría, tratamiento).",
        "args": {
            "centro": "nombre del centro: ensenada, puelche o huito",
            "operacion": "tipo de operación: cosecha, biometría, tratamiento"
        }
    }
}

print("✓ Herramientas del agente definidas.")
print(f"  Herramientas disponibles: {list(tools.keys())}")

✓ Herramientas del agente definidas.
  Herramientas disponibles: ['get_clima_actual', 'get_pronostico_semana', 'evaluar_condiciones_operacion']


In [6]:

def create_system_prompt(tools):
    tool_list = []
    for name, details in tools.items():
        tool_list.append(f"- {name}: {details['description']} Argumentos: {details['args']}")
    
    tools_str = "\n".join(tool_list)
    tool_names = json.dumps(list(tools.keys()))
    
    return f"""Eres un asistente experto en acuicultura para Salmones Camanchaca.
Tu rol es monitorear las condiciones climáticas de los centros de cultivo
(Ensenada, Puelche y Huito) en la región de Los Lagos, Chile,
y apoyar decisiones operativas como cosechas, biometrías y tratamientos.

Sigue estrictamente este formato en cada paso:

Thought (Pensamiento): Razonamiento sobre qué hacer a continuación.
Action (Acción): Herramienta a usar en formato JSON: {{"tool": "nombre", "args": {{argumentos}}}}
Observation (Observación): Resultado de la acción.
Final Answer (Respuesta Final): Respuesta final al operador.

Herramientas disponibles: {tool_names}
{tools_str}"""


def run_agent(user_query, client, tools):
    system_prompt = create_system_prompt(tools)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query}
    ]
    
    print(f"\n--- Agente iniciado para: '{user_query}' ---\n")
    
    for _ in range(5):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0,
            max_tokens=600
        )
        
        text = response.choices[0].message.content
        messages.append({"role": "assistant", "content": text})
        print(f" Agente:\n{text}\n")
        
        if "Final Answer:" in text:
            final_answer = text.split("Final Answer:")[-1].strip()
            print("--- Agente finalizado ---")
            return final_answer
        
        action_match = re.search(r'Action.*?:\s*(\{.*?\})', text, re.DOTALL)
        if action_match:
            try:
                action_json = json.loads(action_match.group(1).strip())
                tool_name   = action_json.get("tool")
                tool_args   = action_json.get("args", {})
                
                if tool_name in tools:
                    observation = tools[tool_name]["function"](tool_args)
                else:
                    observation = f"Herramienta '{tool_name}' no encontrada."
                
                messages.append({"role": "user", "content": f"Observation: {observation}"})
            except Exception as e:
                messages.append({"role": "user", "content": f"Observation: Error - {e}"})
        else:
            print("--- El agente no determinó una acción. ---")
            return text
    
    print("--- Límite de iteraciones alcanzado. ---")
    return "El agente no pudo completar la tarea."


print("✓ Lógica del agente ReAct definida.")

✓ Lógica del agente ReAct definida.


In [10]:
def run_agent(user_query, client, tools):
    system_prompt = create_system_prompt(tools)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query}
    ]
    
    print(f"\n--- Agente iniciado para: '{user_query}' ---\n")
    
    for _ in range(5):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0,
            max_tokens=600
        )
        
        text = response.choices[0].message.content
        messages.append({"role": "assistant", "content": text})
        print(f" Agente:\n{text}\n")
        
        if "Final Answer:" in text:
            final_answer = text.split("Final Answer:")[-1].strip()
            print("---  Agente finalizado ---")
            return final_answer
        
        action_match = re.search(
            r'(?:```json\s*)(\{.*?\})(?:\s*```)|Action.*?:\s*(\{.*?\})',
            text,
            re.DOTALL
        )
        
        if action_match:
            try:
                json_str = action_match.group(1) or action_match.group(2)
                json_str = json_str.strip()
                
                action_json = json.loads(json_str)
                tool_name   = action_json.get("tool")
                tool_args   = action_json.get("args", {})
                
                print(f"    Ejecutando herramienta: {tool_name}")
                print(f"    Argumentos: {tool_args}")
                
                if tool_name in tools:
                    observation = tools[tool_name]["function"](tool_args)
                    print(f"    Resultado: {observation}\n")
                else:
                    observation = f"Herramienta '{tool_name}' no encontrada."
                
                messages.append({
                    "role":    "user",
                    "content": f"Observation: {observation}"
                })
            except json.JSONDecodeError as e:
                messages.append({
                    "role":    "user",
                    "content": f"Observation: Error al parsear JSON - {e}"
                })
            except Exception as e:
                messages.append({
                    "role":    "user",
                    "content": f"Observation: Error - {e}"
                })
        else:
            print("---  El agente no determinó una acción. ---")
            return text
    
    print("---  Límite de iteraciones alcanzado. ---")
    return "El agente no pudo completar la tarea."

print("✓ Función run_agent corregida.")

✓ Función run_agent corregida.


In [11]:


tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "get_clima_actual",
            "description": "Obtiene el clima actual para un centro de cultivo de Camanchaca.",
            "parameters": {
                "type": "object",
                "properties": {
                    "centro": {
                        "type": "string",
                        "description": "Nombre del centro: ensenada, puelche o huito."
                    }
                },
                "required": ["centro"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_pronostico_semana",
            "description": "Obtiene el pronóstico de 7 días para un centro de cultivo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "centro": {
                        "type": "string",
                        "description": "Nombre del centro: ensenada, puelche o huito."
                    }
                },
                "required": ["centro"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "evaluar_condiciones_operacion",
            "description": "Evalúa si las condiciones climáticas son seguras para operar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "centro": {
                        "type": "string",
                        "description": "Nombre del centro: ensenada, puelche o huito."
                    },
                    "operacion": {
                        "type": "string",
                        "description": "Tipo de operación: cosecha, biometría o tratamiento."
                    }
                },
                "required": ["centro", "operacion"]
            }
        }
    }
]

available_tools = {
    "get_clima_actual":             get_clima_actual,
    "get_pronostico_semana":        get_pronostico_semana,
    "evaluar_condiciones_operacion": evaluar_condiciones_operacion
}

print("✓ Herramientas en formato Function Calling definidas.")

✓ Herramientas en formato Function Calling definidas.


In [12]:
def run_agent_function_calling(user_query, client, tools_definition, available_tools):
    messages = [
        {
            "role": "system",
            "content": (
                "Eres un asistente experto en acuicultura para Salmones Camanchaca. "
                "Usas herramientas climáticas para apoyar decisiones operativas en los "
                "centros de cultivo Ensenada, Puelche y Huito en Los Lagos, Chile."
            )
        },
        {"role": "user", "content": user_query}
    ]
    
    print(f"\n---  Iniciando agente Function Calling: '{user_query}' ---\n")
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools_definition,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    messages.append(response_message)
    
    if response_message.tool_calls:
        print(" El modelo ha decidido usar una herramienta...")
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"   Herramienta: {function_name}")
            print(f"   Argumentos:  {function_args}")
            
            function_response = available_tools[function_name](function_args)
            print(f"   Resultado:   {function_response}\n")
            
            messages.append({
                "tool_call_id": tool_call.id,
                "role":         "tool",
                "name":         function_name,
                "content":      function_response
            })
        
        print(" Procesando resultado de la herramienta...")
        second_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )
        return second_response.choices[0].message.content
    else:
        print(" El modelo respondió directamente.")
        return response_message.content


print("✓ Lógica de Function Calling definida.")

✓ Lógica de Function Calling definida.


In [13]:


query_fc = "¿Cuál es el mejor día esta semana para hacer biometría en Puelche?"
respuesta_fc = run_agent_function_calling(query_fc, client, tools_definition, available_tools)
print(f"\n Respuesta Final: {respuesta_fc}")


---  Iniciando agente Function Calling: '¿Cuál es el mejor día esta semana para hacer biometría en Puelche?' ---

 El modelo ha decidido usar una herramienta...
   Herramienta: get_pronostico_semana
   Argumentos:  {'centro': 'puelche'}
   Resultado:   Pronóstico 7 días - Centro Puelche:

2026-05-19: 5.4°C - 12.9°C | Viento: 14.2 km/h | Lluvia: 0.0 mm | Nublado
2026-05-20: 8.9°C - 17.8°C | Viento: 17.6 km/h | Lluvia: 0.0 mm | Despejado
2026-05-21: 6.1°C - 16.6°C | Viento: 13.8 km/h | Lluvia: 0.0 mm | Despejado
2026-05-22: 3.5°C - 14.7°C | Viento: 11.1 km/h | Lluvia: 0.0 mm | Despejado
2026-05-23: 2.2°C - 11.4°C | Viento: 6.9 km/h | Lluvia: 0.0 mm | Nublado
2026-05-24: 4.4°C - 11.9°C | Viento: 12.2 km/h | Lluvia: 0.3 mm | Lluvia
2026-05-25: 3.8°C - 12.4°C | Viento: 13.2 km/h | Lluvia: 0.0 mm | Nublado

 Procesando resultado de la herramienta...

 Respuesta Final: Para realizar la biometría en el centro de cultivo Puelche esta semana, los mejores días son aquellos con condiciones climát

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langgraph.prebuilt import create_react_agent

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0
)

print("✓ LLM LangChain configurado con streaming.")
print(f"Modelo: {llm.model_name}")

NameError: name 'os' is not defined

In [ ]:
@tool
def clima_actual_lc(centro: str) -> str:
    """Obtiene el clima actual para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    return get_clima_actual({"centro": centro})

@tool
def pronostico_semana_lc(centro: str) -> str:
    """Obtiene el pronóstico climático de 7 días para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    return get_pronostico_semana({"centro": centro})

@tool
def evaluar_operacion_lc(centro: str, operacion: str) -> str:
    """Evalúa si las condiciones climáticas son seguras para realizar una operación en Camanchaca.
    centro: ensenada, puelche o huito.
    operacion: cosecha, biometría o tratamiento."""
    return evaluar_condiciones_operacion({"centro": centro, "operacion": operacion})

tools_lc = [clima_actual_lc, pronostico_semana_lc, evaluar_operacion_lc]

prompt_lc = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente experto en acuicultura para Salmones Camanchaca. "
        "Monitoreas las condiciones climáticas de los centros Ensenada, Puelche y Huito "
        "en la región de Los Lagos, Chile, y apoyas decisiones operativas del equipo."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent_lc       = create_openai_tools_agent(llm, tools_lc, prompt_lc)
agent_executor = AgentExecutor(agent=agent_lc, tools=tools_lc, verbose=True)

print("✓ Agente LangChain con herramientas climáticas listo.")

In [ ]:


query_lc = "¿La temperatura esta semana en Huito afectará el FCR del salmón Atlántico?"

response_lc = agent_executor.invoke({
    "input":        query_lc,
    "chat_history": []
})

print(f"\n🏁 Respuesta Final: {response_lc['output']}")

In [ ]:


!pip install crewai crewai-tools -q

In [ ]:
import os
from crewai import Agent, Task, Crew, Process
from crewai_tools import BaseTool
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_BASE"] = os.environ.get("OPENAI_BASE_URL", "")
os.environ["OPENAI_API_KEY"]  = os.environ.get("GITHUB_TOKEN", "")

llm_crew = ChatOpenAI(model="gpt-4o", temperature=0)

class ClimaActualTool(BaseTool):
    name: str        = "Clima Actual Camanchaca"
    description: str = "Obtiene el clima actual de un centro de cultivo. Parámetro: nombre del centro (ensenada, puelche, huito)."
    
    def _run(self, centro: str) -> str:
        return get_clima_actual({"centro": centro})

class PronosticoTool(BaseTool):
    name: str        = "Pronóstico Semanal Camanchaca"
    description: str = "Obtiene el pronóstico de 7 días de un centro de cultivo. Parámetro: nombre del centro."
    
    def _run(self, centro: str) -> str:
        return get_pronostico_semana({"centro": centro})

class EvaluacionOperacionTool(BaseTool):
    name: str        = "Evaluación de Operación Camanchaca"
    description: str = "Evalúa si las condiciones son aptas para una operación. Parámetros: centro y tipo de operación."
    
    def _run(self, centro: str, operacion: str = "cosecha") -> str:
        return evaluar_condiciones_operacion({"centro": centro, "operacion": operacion})

clima_tool      = ClimaActualTool()
pronostico_tool = PronosticoTool()
evaluacion_tool = EvaluacionOperacionTool()

print("✓ Herramientas CrewAI definidas.")

In [ ]:
meteorologo = Agent(
    role="Meteorólogo Acuícola",
    goal="Analizar las condiciones climáticas actuales y el pronóstico semanal de los centros de cultivo de Camanchaca.",
    backstory=(
        "Eres un meteorólogo especializado en condiciones climáticas marítimas del sur de Chile. "
        "Tienes amplia experiencia interpretando datos de viento, temperatura y precipitaciones "
        "para la industria acuícola en la región de Los Lagos."
    ),
    tools=[clima_tool, pronostico_tool],
    llm=llm_crew,
    verbose=True,
    allow_delegation=False
)

supervisor_operaciones = Agent(
    role="Supervisor de Operaciones",
    goal="Evaluar la viabilidad operativa de las actividades en los centros de cultivo basándose en el análisis climático.",
    backstory=(
        "Eres el supervisor de operaciones de Salmones Camanchaca con 10 años de experiencia "
        "en la coordinación de cosechas, biometrías y tratamientos sanitarios. "
        "Tu prioridad es la seguridad del equipo y la eficiencia productiva."
    ),
    tools=[evaluacion_tool],
    llm=llm_crew,
    verbose=True,
    allow_delegation=False
)

print("✓ Agentes CrewAI definidos.")

In [ ]:
tarea_analisis = Task(
    description=(
        "Analiza las condiciones climáticas actuales y el pronóstico de 7 días "
        "para los centros Ensenada y Puelche. Identifica días críticos por viento, "
        "lluvia o temperatura extrema que puedan afectar las operaciones."
    ),
    expected_output=(
        "Un reporte climático con el estado actual y los días de mayor riesgo "
        "para cada centro durante la semana."
    ),
    agent=meteorologo
)

tarea_decision = Task(
    description=(
        "Basándote en el análisis climático del meteorólogo, evalúa si es viable "
        "realizar cosecha en Ensenada y biometría en Puelche esta semana. "
        "Recomienda los días más seguros para cada operación."
    ),
    expected_output=(
        "Un plan operativo semanal con días recomendados para cosecha y biometría, "
        "justificado por las condiciones climáticas."
    ),
    agent=supervisor_operaciones,
    context=[tarea_analisis]
)

crew = Crew(
    agents=[meteorologo, supervisor_operaciones],
    tasks=[tarea_analisis, tarea_decision],
    process=Process.sequential,
    verbose=True
)

print("✓ Crew de Camanchaca ensamblado.")
print(f"  Agentes: {len(crew.agents)}")
print(f"  Tareas:  {len(crew.tasks)}")

In [ ]:


try:
    print(" Iniciando equipo de agentes Camanchaca...\n")
    resultado = crew.kickoff()
    
    print("\n" + "="*60)
    print(" REPORTE FINAL DEL EQUIPO CAMANCHACA")
    print("="*60)
    print(resultado)
except Exception as e:
    print(f" Error: {e}")
    import traceback
    traceback.print_exc()